# Kvasir-SEG inference

Notebook para cargar un checkpoint entrenado y ejecutar inferencia sobre una imagen.

In [ ]:
from pathlib import Path
import sys
import zipfile

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from inference import load_model, predict_mask, save_prediction, show_prediction

checkpoint_path = PROJECT_ROOT / "outputs/checkpoints/unet_kvasir.pt"
image_path = PROJECT_ROOT / "data/Kvasir-SEG/images/cju0qkwl35piu0993l0dewei2.jpg"
output_path = PROJECT_ROOT / "outputs/predictions/notebook_prediction.png"

if not image_path.exists():
    zip_path = PROJECT_ROOT / "data/kvasir-seg.zip"
    sample_dir = PROJECT_ROOT / "outputs/predictions"
    sample_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        image_name = next(name for name in archive.namelist() if "/images/" in name and Path(name).suffix.lower() in {".jpg", ".jpeg", ".png"})
        image_path = sample_dir / Path(image_name).name
        image_path.write_bytes(archive.read(image_name))

checkpoint_path, image_path, output_path

Puedes cambiar `image_path` por cualquier imagen suelta. Si solo existe `data/kvasir-seg.zip`, la celda anterior extrae una imagen de ejemplo en `outputs/predictions/`.

In [ ]:
model, device = load_model(checkpoint_path)
probability, mask = predict_mask(model, image_path, device, image_size=256, threshold=0.5)
save_prediction(mask, output_path)
fig = show_prediction(image_path, probability, mask)
print(f"Saved prediction to {output_path}")